# 04 — Improvement round (v1 → v2)

Decides, **without touching the test set**, whether the v2 features help.

**New in v2** (all causal, all covered by `tests/test_features.py`):
- `block_hour` + *block-so-far* stats (mean/max Newell, mean V·Bs, min Bz since the current Kp block started) — v1's error was highest in the first hour of each block
- pressure dynamics (3 h mean/max, 1 h change) — pressure was v1's top feature
- 12 h / 24 h Newell coupling means
- wider HistGB search (20 settings instead of 8)

**Leakage protocol**
1. Tuning: `TimeSeriesSplit(5)` CV inside 1998–2016 only.
2. Decisions: CV + validation (2017–2019) only. **Test (2020+) is not used in this notebook.**
3. Rule fixed in advance: adopt v2 only if **both** CV RMSE and validation RMSE improve.
4. Then `03_modeling.ipynb` is re-run once to report v2 on test, next to the archived v1 results (`reports/v1/`).

Runtime ≈ 15–20 min. Kernel: **Python 3.12 (tormenta)**.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd

from src import config, train, evaluate as ev
from src.features import kp_history_features
from src.predict import load_bundle
from src.preprocessing import load_processed

FAST = False
pd.set_option("display.precision", 3)

## 1. Feature sets

In [2]:
X, y, persistence = train.load_xy()
masks = train.split_masks(X.index)
v1_cols = load_bundle(config.MODELS_DIR / "tormenta_v20261028.joblib")["features"]
new_cols = [c for c in X.columns if c not in v1_cols]
print(f"v1: {len(v1_cols)} features | v2: {X.shape[1]} features | new: {new_cols}")
assert set(v1_cols) <= set(X.columns)

v1: 56 features | v2: 66 features | new: ['newell_mean12h', 'newell_mean24h', 'pressure_mean3h', 'pressure_max3h', 'pressure_change1h', 'block_hour', 'newell_block_mean', 'vbs_block_mean', 'newell_block_max', 'bz_block_min']


## 2. HistGB v1 vs v2 — same search, same CV, judged on CV + validation

In [3]:
tuned = {}
for label, cols in [("v1", v1_cols), ("v2", list(X.columns))]:
    print(f"--- {label} ---")
    tuned[label] = train.tune_models(X[cols], y, masks, fast=FAST, names=["HistGB"])["HistGB"]
    tuned[label]["cols"] = cols

--- v1 ---
HistGB             CV RMSE 0.572  (  424 s)  {'model__l2_regularization': np.float64(0.0062353771356731605), 'model__learning_rate': np.float64(0.030315935562393058), 'model__max_iter': 700, 'model__max_leaf_nodes': 63, 'model__min_samples_leaf': 20}
--- v2 ---
HistGB             CV RMSE 0.561  (  438 s)  {'model__l2_regularization': np.float64(0.2801635158716261), 'model__learning_rate': np.float64(0.039088721491346144), 'model__max_iter': 700, 'model__max_leaf_nodes': 31, 'model__min_samples_leaf': 50}


In [4]:
val = masks["val"]
val_preds = {f"HistGB {k}": r["estimator"].predict(X.loc[val, r["cols"]]) for k, r in tuned.items()}
cmp = ev.metrics_table(y[val], val_preds)
cmp["cv_rmse"] = [tuned["v1"]["cv_rmse"], tuned["v2"]["cv_rmse"]]
cmp.to_csv(config.METRICS_DIR / "v1_vs_v2_validation.csv")
display(cmp.drop(columns="n"))
by_pos = ev.rmse_by_block_hour(y[val], val_preds)
by_pos.to_csv(config.METRICS_DIR / "v1_vs_v2_block_hour.csv")
by_pos

,rmse,mae,r2,storm_rmse,storm_mae,cv_rmse
HistGB v1,0.529,0.409,0.809,0.889,0.710,0.572
HistGB v2,0.522,0.404,0.814,0.853,0.674,0.561


hour 1 of block  hour 2 of block  hour 3 of block
HistGB v1 all              0.564            0.508            0.514
          storm            1.020            0.843            0.789
HistGB v2 all              0.556            0.508            0.500
          storm            0.982            0.819            0.740

## 3. Decision (rule fixed in advance)

In [5]:
cv_better = tuned["v2"]["cv_rmse"] < tuned["v1"]["cv_rmse"]
val_better = cmp.loc["HistGB v2", "rmse"] < cmp.loc["HistGB v1", "rmse"]
ADOPT_V2 = bool(cv_better and val_better)
print(f"CV improves: {cv_better} | validation improves: {val_better} → adopt v2: {ADOPT_V2}")
print("Best v2 HistGB settings:", tuned["v2"]["best_params"])

CV improves: True | validation improves: True → adopt v2: True
Best v2 HistGB settings: {'model__l2_regularization': np.float64(0.2801635158716261), 'model__learning_rate': np.float64(0.039088721491346144), 'model__max_iter': 700, 'model__max_leaf_nodes': 31, 'model__min_samples_leaf': 50}


## 4. Ablation: what if the model could also see past Kp?
**Not for production** (our design rule excludes geomagnetic inputs). Adds Kp of the two previous *complete* blocks and the previous-24 h mean — never the current block (unit-tested). Shows the upper bound that autoregression would add, putting the solar-wind-only result in context. Validation years only.

In [6]:
kp_feats = kp_history_features(load_processed()[config.TARGET], X.index)
abl, _ = train.kp_history_ablation(tuned["v2"]["best_params"], X, y, masks, kp_feats, fast=FAST)
abl.loc["Persistence (baseline)"] = ev.metrics(y[val], persistence[val])
abl.to_csv(config.METRICS_DIR / "ablation_kp_history.csv")
abl.drop(columns="n")

,rmse,mae,r2,storm_rmse,storm_mae
HistGB (solar wind only),0.522,0.404,0.814,0.853,0.674
HistGB + past Kp (ablation),0.512,0.395,0.821,0.832,0.653
Persistence (baseline),0.819,0.606,0.542,1.355,1.034


## Next
If `ADOPT_V2` is True, re-run **`03_modeling.ipynb`** (all models, v2 features) — it reports test once, compares against the v1 model on the gate holdout and only promotes v2 if it is at least as good. If False, tell Claude — the v2 features get reverted.